# 🎓 College Buddy - Google Colab Setup 🚀

This notebook runs the College Buddy Chatbot Backend on Google's T4 GPU.

### Features:
- **GPU-Accelerated**: Uses T4 GPU for fast inference (~2-5s).
- **Model**: `llama3.2:3b` via Ollama.
- **Public URL**: Exposes backend via Ngrok for frontend connection.

### ⚠️ IMPORTANT: Runtime Setup
Make sure you are using a **GPU Runtime**:
1. Click **Runtime** > **Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

In [ ]:
# Verify GPU is available
!nvidia-smi

## 1. Install Dependencies
Installing Ollama, ChromaDB, and required Python libraries.

**Note**: This may take 2-3 minutes.

In [ ]:
# Fix for ChromaDB in Colab
!pip install -q pysqlite3-binary
import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
print("✅ SQLite fixed")

import os
import pkg_resources

# 🛑 CRITICAL DEPENDENCY FIX 
# Check if we need to fix numpy (Colab often pre-installs Numpy 2.x)
needs_restart = False
try:
    np_ver = pkg_resources.get_distribution("numpy").version
    if np_ver.startswith("2"):
        print(f"⚠️ Detected Numpy {np_ver} (Incompatible). Installing Numpy 1.x...")
        needs_restart = True
except:
    pass

if needs_restart:
    # Uninstall conflicting packages first
    !pip uninstall -y numpy pandas
    
    # Install compatible versions
    !pip install "numpy<2.0.0" "pandas<2.2.0" fastapi uvicorn pyngrok nest_asyncio sentence-transformers chromadb langchain langchain-community langchain-ollama openpyxl requests==2.32.3 opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp langchain-huggingface
    
    print("✅ Python dependencies installed (Numpy 1.x compatible)")
    print("\n🛑 RESTARTING RUNTIME TO APPLY FIXES... (This is normal!)")
    print("👉 Please RUN THIS CELL AGAIN after the restart!")
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)  # Kill runtime to force reload of numpy
else:
    # Standard install if numpy is already 1.x
    print("✅ Numpy is compatible. checking other dependencies...")
    !pip install "numpy<2.0.0" "pandas<2.2.0" fastapi uvicorn pyngrok nest_asyncio sentence-transformers chromadb langchain langchain-community langchain-ollama openpyxl requests==2.32.3 opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp langchain-huggingface
    print("✅ Python dependencies installed")

# Install zstd (required for Ollama extraction)
!apt-get update -qq && apt-get install -qq -y zstd > /dev/null 2>&1
print("✅ zstd installed")

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed")

## 2. Start Ollama and Pull Model
Starting Ollama server and downloading `llama3.2:3b` (~2GB download).

In [ ]:
import subprocess
import time
import os

# Check if Ollama is installed
if not os.path.exists('/usr/local/bin/ollama') and not os.path.exists('/usr/bin/ollama'):
    print("❌ Ollama not found! Please run the 'Install Dependencies' cell above.")
else:
    print("✅ Ollama found, starting server...")
    
    # Start Ollama in background
    process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)  # Wait for server to start

    print("⏳ Pulling llama3.2:3b model (this may take 2-3 minutes)...")
    !ollama pull llama3.2:3b
    print("✅ Model ready!")

## 3. Upload Codebase

### Option A: Upload via File Explorer (Easiest)
1. Click the **📁 Files** icon on the left sidebar
2. Drag and drop these items from your local project:
   - `backend.py`
   - `app/` (entire folder)

### Option B: Clone from GitHub
Uncomment and run the cell below if your code is on GitHub:

In [ ]:
# Option B: Clone from GitHub
!git clone https://github.com/VijayKiran-2004/College-Chat-Bot.git
%cd College-Chat-Bot

# Verify file structure
print("Checking uploaded files...")
!ls -la

import os
if os.path.exists('backend.py') and os.path.exists('app'):
    print("✅ Core files uploaded successfully!")
else:
    print("❌ Missing 'backend.py' or 'app/' folder. Please upload them using Option A.")

## 4. Download Database Files

Since the database files are now in the GitHub repo, just run the cell below to pull them.


In [ ]:
# Pull latest changes (including database files)
!git pull

import os

# Create necessary directories
!mkdir -p app/database/vectordb/chroma
!mkdir -p logs
print("✅ Created missing directories")

# Verify critical files
corpus_exists = os.path.exists('app/database/vectordb/corpus_ultrarag.jsonl')
db_exists = os.path.exists('app/database/students.db')

print("\n📋 File Status:")
print(f"  {'✅' if corpus_exists else '❌'} corpus_ultrarag.jsonl")
print(f"  {'✅' if db_exists else '❌'} students.db")

if not corpus_exists or not db_exists:
    print("\n⚠️ WARNING: Missing files even after git pull.")
    print("   Please check if they were pushed to the repo.")
else:
    print("\n✅ All required files present!")

## 5. Setup Ngrok

### Get Your Free Ngrok Token:
1. Visit: https://dashboard.ngrok.com/get-started/your-authtoken
2. Sign up (free)
3. Copy your token
4. Paste it in the cell below

In [ ]:
# ⚠️ REPLACE WITH YOUR NGROK TOKEN
# Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "38AFZ9ehRUeryGuOWMeyCuQ4dvm_3yEgMYUjpWJsY4hTChMVf"

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("❌ ERROR: Please set your Ngrok token above!")
    print("   Get one for free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok configured successfully!")

## 6. Start Backend Server 🚀

This will:
1. Initialize the RAG system (ChromaDB + Ollama)
2. Start the FastAPI server
3. Create a public URL via Ngrok

**Copy the public URL** and use it in your frontend's `API_BASE_URL`.

In [ ]:
import os
import uvicorn
import nest_asyncio
from pyngrok import ngrok

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Kill existing tunnels
ngrok.kill()

# Start Ngrok tunnel
try:
    public_url = ngrok.connect(8000).public_url
    print("="*70)
    print("🚀 BACKEND IS LIVE!")
    print("="*70)
    print(f"\n📋 Public URL: {public_url}")
    print(f"\n🔗 Use this URL in your frontend:")
    print(f"   const API_BASE_URL = '{public_url}';")
    print("\n" + "="*70)
except Exception as e:
    print(f"❌ Ngrok Error: {e}")
    print("   Make sure you set your token in the cell above!")

# Verify we are in the right directory before starting
if not os.path.exists('backend.py'):
    print("❌ ERROR: 'backend.py' not found!")
    print("   Did you run Cell 4 to clone the repository?")
    print("   Trying to find it...")
    if os.path.exists('College-Chat-Bot/backend.py'):
        os.chdir('College-Chat-Bot')
        print("✅ Found it in College-Chat-Bot/ folder. Switched directory.")
    else:
        raise FileNotFoundError("Please run Cell 4 to clone the code first!")

# Start FastAPI server correctly in Colab
try:
    from backend import app
    print("\n⏳ Starting server (initialization may take 30-60 seconds)...\n")
    
    # Run using Config + Server to support async loop
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()
    
except ImportError:
    print("❌ ERROR: 'backend.py' not found!")
    print("   Please run Cell 4 to clone your code.")
except Exception as e:
    print(f"❌ Server Error: {e}")
    import traceback
    traceback.print_exc()
    print("\nTroubleshooting:")
    print("  1. Ensure all database files are uploaded")
    print("  2. Check that Ollama is running (run cell 2 again)")
    print("  3. Verify GPU is available (run cell 1)")

## 🎉 Success!

If the server started successfully, you should see:
- `✓ Backend initialized in X seconds`
- The server is now ready to accept requests at the public URL above

### Next Steps:
1. Copy the public URL from above
2. Update your frontend's `API_BASE_URL` variable
3. Open your frontend and test the chatbot!

### Troubleshooting:
- If you see errors, scroll up to check which step failed
- Common issues:
  - Missing database files → Re-upload them
  - Ollama not running → Re-run cell 2
  - Out of memory → Restart runtime and try again